In [1]:
# Run this each time 
!pip install -U transformers accelerate pandas openpyxl

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 102.0 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 

In [2]:
import os
import pandas as pd
import json
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# =========================
# 1. Local model setup
# =========================
model_name = "Qwen/Qwen2.5-1.5B-Instruct"


cache_dir = "/content/drive/MyDrive/hf_cache"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=cache_dir
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
    cache_dir=cache_dir,
    low_cpu_mem_usage=True
)

model.eval()

print("Model loaded successfully.")
print("CUDA available:", torch.cuda.is_available())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully.
CUDA available: True


In [26]:
# 2. Read Excel
# =========================
input_file = "/content/drive/MyDrive/sample50persona.xlsx"

df_personas = pd.read_excel(
    input_file,
    skiprows=2
)

df_personas.columns = df_personas.columns.str.strip()

print("Loaded personas:", len(df_personas))
print(df_personas.head())
print(df_personas.columns)


Loaded personas: 50
   sample_id sample_code  persona_id persona_code            age_group  \
0          1        S001           8         P008  Aged 18 to 24 years   
1          2        S002          16         P016  Aged 18 to 24 years   
2          3        S003          21         P021  Aged 18 to 24 years   
3          4        S004          41         P041  Aged 18 to 24 years   
4          5        S005          63         P063  Aged 18 to 24 years   

     region income household_composition                    tech_adoption  \
0   England    low  Couple with children                Selective adopter   
1   England    mid   Couple, no children                    Early adopter   
2   England    mid  Couple with children  Late adopter/conservative users   
3  Scotland    low   Couple, no children                Selective adopter   
4  Scotland   high   Single, no children  Late adopter/conservative users   

                                     persona_summary  
0  Aged 18 to 24 

In [27]:
# 3. Prompt
# =========================
def build_prompt(row, bank_name="First Direct"):
    return f"""
You are simulating a UK retail banking customer.

Persona:
- Age: {row['age_group']}
- Region: {row['region']}
- Income: {row['income']}
- Household: {row['household_composition']}
- Technology adoption: {row['tech_adoption']}

Task:
Estimate how likely this customer would be to recommend {bank_name} to a friend or colleague, on a scale from 0 to 10.

Important rules:
- Base your answer on plausible general perceptions of the bank in the UK retail banking market.
- Do not invent specific product facts or service limitations unless you are certain.
- If you are uncertain about a feature, do not mention it as a reason.
- Give a realistic score, not an extreme score unless clearly justified.
- For household composition, do not assume the bank lacks joint-account functionality unless explicitly stated.
- The reason should reflect likely customer preferences or priorities, not unverifiable factual claims about the bank.


Return ONLY valid JSON:
{{
  "score": <INTEGER>,
  "reason": "one short sentence clearly linked to persona traits"
}}
""".strip()

In [28]:
# 4. Local generation function
# =========================
def generate_response(prompt):
    messages = [
        {
            "role": "system",
            "content": "Return only one valid JSON object. No markdown. No explanation."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.5,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    raw_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    raw_text = raw_text.replace("```json", "").replace("```", "").strip()

    # Extract the first complete JSON object to avoid extra model output
    start = raw_text.find("{")
    end = raw_text.find("}", start)

    if start != -1 and end != -1:
        return raw_text[start:end + 1]

    return raw_text

In [29]:
# 5. JSON cleaning
# =========================
def extract_json(text):
    if not text:
        return None

    text = text.strip()
    text = text.replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(text)
    except:
        pass

    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except:
            return None

    return None


In [30]:
# 6. NPS classification
# =========================
def get_nps_category(score):
    if pd.isna(score):
        return None
    if score >= 9:
        return "Promoter"
    elif score >= 7:
        return "Passive"
    else:
        return "Detractor"


In [23]:
# 7. Test the first row
# =========================
row = df_personas.iloc[0]
prompt = build_prompt(row)
raw_text = generate_response(prompt)
parsed = extract_json(raw_text)

print("RAW:")
print(raw_text)
print("PARSED:")
print(parsed)

RAW:
{
  "score": 3,
  "reason": "Age and income suggest financial conservatism, selective technology adoption implies they might prefer more traditional services."
}
PARSED:
{'score': 3, 'reason': 'Age and income suggest financial conservatism, selective technology adoption implies they might prefer more traditional services.'}


In [31]:
# =========================
# Single-persona stability test Single-persona stability test: run first persona 10 times
# =========================

import pandas as pd
import time

row = df_personas.iloc[0]
n_runs = 10

stability_results = []

for run_id in range(1, n_runs + 1):
    prompt = build_prompt(row)
    raw_text = generate_response(prompt)
    parsed = extract_json(raw_text)

    score = parsed.get("score") if isinstance(parsed, dict) else None
    reason = parsed.get("reason") if isinstance(parsed, dict) else None

    stability_results.append({
        "run_id": run_id,
        "persona_id": row.get("persona_id", 0),
        "score": score,
        "reason": reason,
        "raw_text": raw_text
    })

    print(f"\n===== RUN {run_id} =====")
    print("RAW:")
    print(raw_text)
    print("PARSED:")
    print(parsed)

    time.sleep(1)

df_stability = pd.DataFrame(stability_results)

print("\n===== Stability Summary =====")
print(df_stability[["run_id", "score", "reason"]])

print("\nScore statistics:")
print(df_stability["score"].describe())


===== RUN 1 =====
RAW:
{
  "score": 3,
  "reason": "Age and income suggest financial literacy but limited spending power; selective technology use implies cautious behavior."
}
PARSED:
{'score': 3, 'reason': 'Age and income suggest financial literacy but limited spending power; selective technology use implies cautious behavior.'}

===== RUN 2 =====
RAW:
{
  "score": 6,
  "reason": "The customer is selective adopter and has limited income, which may lead them to prefer banks that offer simple services."
}
PARSED:
{'score': 6, 'reason': 'The customer is selective adopter and has limited income, which may lead them to prefer banks that offer simple services.'}

===== RUN 3 =====
RAW:
{
  "score": 5,
  "reason": "The customer is selective adopter and has low income, which makes them more cautious when choosing a bank."
}
PARSED:
{'score': 5, 'reason': 'The customer is selective adopter and has low income, which makes them more cautious when choosing a bank.'}

===== RUN 4 =====
RAW:
{
  

In [32]:
# Optional: save results
output_file = "/content/drive/MyDrive/single_persona_stability_test.csv"
df_stability.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"\nSaved to: {output_file}")


Saved to: /content/drive/MyDrive/single_persona_stability_test.csv


In [33]:
# =========================
# Cross-persona validation Cross-persona validation: 5 contrasting personas
# =========================

import pandas as pd
import time

test_personas = [
    {
        "age_group": "Aged 18 to 24 years",
        "region": "England",
        "income": "low",
        "household_composition": "Single, no children",
        "tech_adoption": "Early adopter"
    },
    {
        "age_group": "Aged 60 and over",
        "region": "Wales",
        "income": "low",
        "household_composition": "Couple, no children",
        "tech_adoption": "Late adopter/conservative users"
    },
    {
        "age_group": "Aged 35 to 44 years",
        "region": "England",
        "income": "mid",
        "household_composition": "Single parent/lone parent",
        "tech_adoption": "Selective adopter"
    },
    {
        "age_group": "Aged 45 to 59 years",
        "region": "Scotland",
        "income": "high",
        "household_composition": "Couple, no children",
        "tech_adoption": "Selective adopter"
    },
    {
        "age_group": "Aged 25 to 34 years",
        "region": "England",
        "income": "mid",
        "household_composition": "Couple with children",
        "tech_adoption": "Selective adopter"
    }
]

cross_results = []

for i, persona in enumerate(test_personas, start=1):
    row = pd.Series(persona)
    prompt = build_prompt(row)

    raw_text = generate_response(prompt)
    parsed = extract_json(raw_text)

    score = parsed.get("score") if isinstance(parsed, dict) else None
    reason = parsed.get("reason") if isinstance(parsed, dict) else None

    cross_results.append({
        "persona_no": i,
        "age_group": persona["age_group"],
        "region": persona["region"],
        "income": persona["income"],
        "household_composition": persona["household_composition"],
        "tech_adoption": persona["tech_adoption"],
        "score": score,
        "reason": reason,
        "raw_text": raw_text
    })

    print(f"\n===== PERSONA {i} =====")
    print(persona)
    print("RAW:")
    print(raw_text)
    print("PARSED:")
    print(parsed)

    time.sleep(1)

df_cross = pd.DataFrame(cross_results)

print("\n===== Cross-Persona Validation Summary =====")
print(df_cross[[
    "persona_no",
    "age_group",
    "region",
    "income",
    "household_composition",
    "tech_adoption",
    "score",
    "reason"
]])

# Optional: map scores to NPS categories
def map_nps_group(score):
    if pd.isna(score):
        return None
    score = int(score)
    if score <= 6:
        return "Detractor"
    elif score <= 8:
        return "Passive"
    else:
        return "Promoter"

df_cross["nps_group"] = df_cross["score"].apply(map_nps_group)

print("\nNPS group counts:")
print(df_cross["nps_group"].value_counts(dropna=False))


===== PERSONA 1 =====
{'age_group': 'Aged 18 to 24 years', 'region': 'England', 'income': 'low', 'household_composition': 'Single, no children', 'tech_adoption': 'Early adopter'}
RAW:
{
  "score": 7,
  "reason": "This age group is generally more tech-savvy and may prefer digital services."
}
PARSED:
{'score': 7, 'reason': 'This age group is generally more tech-savvy and may prefer digital services.'}

===== PERSONA 2 =====
{'age_group': 'Aged 60 and over', 'region': 'Wales', 'income': 'low', 'household_composition': 'Couple, no children', 'tech_adoption': 'Late adopter/conservative users'}
RAW:
{
  "score": 5,
  "reason": "Late adopter/conservative user prefers established brands like Barclays."
}
PARSED:
{'score': 5, 'reason': 'Late adopter/conservative user prefers established brands like Barclays.'}

===== PERSONA 3 =====
{'age_group': 'Aged 35 to 44 years', 'region': 'England', 'income': 'mid', 'household_composition': 'Single parent/lone parent', 'tech_adoption': 'Selective adopt

In [35]:
# Optional: save results
output_file = "/content/drive/MyDrive/cross_persona_validation.csv"
df_cross.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"\nSaved to: {output_file}")


Saved to: /content/drive/MyDrive/cross_persona_validation.csv


In [24]:
# 8. Run 50 rows in batch
# =========================
results = []

for i, row in df_personas.iterrows():
    prompt = build_prompt(row)

    try:
        raw_text = generate_response(prompt)
        parsed = extract_json(raw_text)

        if parsed:
            score = parsed.get("score", None)
            reason = parsed.get("reason", "")
        else:
            score = None
            reason = "JSON_PARSE_ERROR"

        try:
            score = int(score) if score is not None else None
            if score is not None and not (0 <= score <= 10):
                score = None
                reason = "INVALID_SCORE"
        except:
            score = None
            reason = "INVALID_SCORE_TYPE"

        results.append({
            "sample_id": row.get("sample_id", ""),
            "persona_id": row.get("persona_id", i + 1),
            "age_group": row.get("age_group", ""),
            "region": row.get("region", ""),
            "income": row.get("income", ""),
            "household_composition": row.get("household_composition", ""),
            "tech_adoption": row.get("tech_adoption", ""),
            "score": score,
            "reason": reason,
            "raw_response": raw_text
        })

        print(f"{i+1}/{len(df_personas)} done | score={score}")

    except Exception as e:
        print(f"Error at {i+1}: {e}")

        results.append({
            "sample_id": row.get("sample_id", ""),
            "persona_id": row.get("persona_id", i + 1),
            "age_group": row.get("age_group", ""),
            "region": row.get("region", ""),
            "income": row.get("income", ""),
            "household_composition": row.get("household_composition", ""),
            "tech_adoption": row.get("tech_adoption", ""),
            "score": None,
            "reason": f"LOCAL_MODEL_ERROR: {e}",
            "raw_response": ""
        })

1/50 done | score=3
2/50 done | score=7
3/50 done | score=6
4/50 done | score=7
5/50 done | score=7
6/50 done | score=7
7/50 done | score=7
8/50 done | score=6
9/50 done | score=7
10/50 done | score=7
11/50 done | score=7
12/50 done | score=8
13/50 done | score=7
14/50 done | score=7
15/50 done | score=6
16/50 done | score=7
17/50 done | score=8
18/50 done | score=7
19/50 done | score=7
20/50 done | score=7
21/50 done | score=7
22/50 done | score=7
23/50 done | score=8
24/50 done | score=7
25/50 done | score=7
26/50 done | score=6
27/50 done | score=8
28/50 done | score=8
29/50 done | score=7
30/50 done | score=7
31/50 done | score=7
32/50 done | score=8
33/50 done | score=8
34/50 done | score=7
35/50 done | score=7
36/50 done | score=8
37/50 done | score=6
38/50 done | score=7
39/50 done | score=3
40/50 done | score=7
41/50 done | score=7
42/50 done | score=8
43/50 done | score=3
44/50 done | score=8
45/50 done | score=8
46/50 done | score=5
47/50 done | score=7
48/50 done | score=7
4

In [31]:
# 9. Save results
# =========================
df_results = pd.DataFrame(results)
df_results["category"] = df_results["score"].apply(get_nps_category)

output_file = "/content/drive/MyDrive/HSBC_nps_results_50_qwen_1.5b.csv"
df_results.to_csv(output_file, index=False, encoding="utf-8-sig")

print("Saved to:", output_file)


Saved to: /content/drive/MyDrive/HSBC_nps_results_50_qwen_1.5b.csv


In [25]:
df_results = pd.DataFrame(results)
df_results["category"] = df_results["score"].apply(get_nps_category)

# 10. Calculate NPS
# =========================
valid = df_results["score"].notna().sum()
promoters = (df_results["category"] == "Promoter").sum()
passives = (df_results["category"] == "Passive").sum()
detractors = (df_results["category"] == "Detractor").sum()

nps = (promoters - detractors) / valid * 100 if valid > 0 else None

print("\n===== SUMMARY =====")
print("Valid:", valid)
print("Promoters:", promoters)
print("Passives:", passives)
print("Detractors:", detractors)
print("NPS:", nps)

print("\nScore distribution:")
print(df_results["score"].value_counts().sort_index())


===== SUMMARY =====
Valid: 50
Promoters: 0
Passives: 41
Detractors: 9
NPS: -18.0

Score distribution:
score
3     3
5     1
6     5
7    30
8    11
Name: count, dtype: int64
